In [0]:
df = spark.read.format('csv').option("header","true")\
    .option("inferSchema","true")\
    .load('/Volumes/mano_catalog/source/vol/sample/trips.csv')\
        

display(df)

In [0]:
from pyspark.sql.functions import sum

df.groupBy("payment_method").agg(sum("fare_amount")).where(sum("fare_amount")>6000).show()

In [0]:
df.write.format('csv').mode('overwrite').save('/Volumes/mano_catalog/source/vol/sample_out/')

In [0]:
df.write.saveAsTable('mano_catalog.silver.trips')

In [0]:
df.write.format('delta').mode('append').partitionBy('payment_method').saveAsTable('mano_catalog.silver.trips')

In [0]:
d1 = spark.read.table('mano_catalog.silver.trips')

In [0]:
d2 = df.selectExpr('payment_method','trip_status as tripStatus')
display(d2)

In [0]:
from pyspark.sql.functions import round,lit

d3 = d1.withColumn('Duration',d1.trip_end_time-d1.trip_start_time)\
    .withColumn('rate', round(d1.fare_amount/d1.distance_km, 2))\
    .withColumn('table_name',lit('trips'))

display(d3)

In [0]:
from pyspark.sql.functions import round,lit,col
d4 = d3.filter(d3.trip_status == 'Completed').orderBy("fare_amount")
#d4 = d3.filter(d3.trip_status == 'Completed').orderBy(col("fare_amount").desc())
display(d4)

In [0]:
from pyspark.sql.functions import round,lit,col,min,max
d1.select(min('fare_amount'),max('fare_amount')).display()

In [0]:
from pyspark.sql.functions import round,lit,col,min,max,count
d1.groupBy('payment_method').agg(count('payment_method'),min('fare_amount'),max('fare_amount')).display()

In [0]:
from pyspark.sql.functions import round,lit,col,min,max,count,collect_set,collect_list
d1.groupBy('payment_method').agg(collect_set('start_location')).display()

In [0]:
from pyspark.sql.functions import row_number,rank,col
from pyspark.sql.window import Window

win_spec = Window.partitionBy('payment_method').orderBy(col('fare_amount').desc())
d6 = d1.withColumn('rn',row_number().over(win_spec))
d6.select('payment_method','customer_id','fare_amount','rn').filter(col('rn')<=5).display()


In [0]:
from pyspark.sql.functions import rank,dense_rank,col
from pyspark.sql.window import Window

win_spec = Window.partitionBy('payment_method').orderBy(col('fare_amount').desc())
d6 = d1.withColumn('rn',rank().over(win_spec))\
    .withColumn('rn_dense',dense_rank().over(win_spec))
d6.select('payment_method','customer_id','fare_amount','rn','rn_dense').display()
